# Chapter 3: Pandas Advanced — Time Series, Reshaping & Production Pipelines

### Why This Matters

Chapter 2 covered what you'll use in nearly every analysis. This chapter covers what separates a competent analyst from a genuinely proficient one: working with dates properly, reshaping data between "long" and "wide" formats, and writing transformations as clean, readable, re-runnable pipelines rather than a scattered mess of intermediate variables.

### 3.1 Time Series: Date Ranges and Rolling Windows

In [ ]:
dates = pd.date_range("2026-01-01", periods=12, freq="W")     # 12 weekly timestamps starting Jan 1, 2026
df = pd.DataFrame({
    "date": dates,
    "sales": [120, 135, 128, 160, 172, 155, 180, 190, 205, 198, 215, 225]
})
df["rolling_4w"] = df["sales"].rolling(4).mean()     # a 4-period MOVING AVERAGE
print(df)

`pd.date_range` is how you generate a sequence of timestamps programmatically — `freq="W"` means weekly; other common frequencies are `"D"` (daily), `"M"` (month-end), `"Q"` (quarter-end), `"H"` (hourly). This is invaluable when you need to build a complete date scaffold to merge sparse real transaction data against (so months with zero sales show up as 0, not simply as an absent row).

`.rolling(4).mean()` computes a **rolling** (moving) average across a 4-row window — at each row, it looks back at that row and the 3 before it, averages them, and that becomes the smoothed value. Notice the first 3 rows will be `NaN`, since there aren't yet 4 rows of history to average. Rolling windows are the standard technique for smoothing out noisy time series data to reveal an underlying trend — you'll see this exact idea reappear as a stated feature in the Statsmodels time-series material in Volume 04.

💡 Related and equally useful: `.expanding().mean()` computes a running average using *all* data up to that point (growing window) rather than a fixed-size window; `.pct_change()` computes the percentage change from the previous row directly, which is usually cleaner than manually computing `(current - previous) / previous`.

### 3.2 Pivoting and Melting — Long vs. Wide Data

In [ ]:
long = pd.DataFrame({
    "region": ["East", "East", "West", "West"],
    "month":  ["Jan", "Feb", "Jan", "Feb"],
    "sales":  [100, 120, 90, 110]
})

wide = long.pivot_table(index="region", columns="month", values="sales")
print(wide)

back_to_long = wide.reset_index().melt(id_vars="region", var_name="month", value_name="sales")
print(back_to_long)

**"Long" format** has one row per observation (region+month+sales as three separate columns, many rows). **"Wide" format** has one row per entity with separate columns for each category (one row per region, with a separate column for each month). Neither is universally "correct" — which you want depends entirely on the tool downstream: most plotting and modeling functions (including scikit-learn) prefer **long** format, while human-readable reports and Excel-style summary tables are usually **wide**. `.pivot_table()` converts long → wide; `.melt()` converts wide → long. Being fluent in converting between them on demand is a skill you'll lean on constantly once you start building dashboards or feeding data into visualization libraries in Volume 03.

### 3.3 Method Chaining — Writing Clean, Production-Style Pipelines

In [ ]:
raw = pd.DataFrame({"name": ["A", "B", "C"], "sales": ["100", "200", "150"]})   # sales loaded as TEXT — common real-world issue

clean = (
    raw
    .assign(
        name=lambda d: d["name"].str.strip().str.upper(),
        sales=lambda d: pd.to_numeric(d["sales"])
    )
    .assign(sales_growth=lambda d: d["sales"].pct_change())
)
print(clean)

`.assign()` creates one or more new/modified columns and returns a *new* DataFrame, which makes it perfectly suited to **chaining** — stringing multiple transformation steps together in a single readable expression, wrapped in parentheses so each `.method()` call can sit on its own line. The `lambda d: ...` pattern inside `.assign()` refers to "the DataFrame as it exists at this exact point in the chain" — which matters because the second `.assign()` call's `sales_growth` calculation needs `sales` to *already* be numeric, which only happened in the first `.assign()` call moments earlier in the same chain.

💡 **Why bother chaining instead of separate lines with intermediate variables?** Two reasons professional codebases favor it: (1) it eliminates a cascade of throwaway variable names (`df1`, `df2`, `df_clean`, `df_final`...) that clutter a notebook and are easy to accidentally reuse in the wrong order; (2) it reads top-to-bottom as a linear narrative of "here's what happens to the data, in order," which is far easier for someone else (or you, in six months) to audit. That said, chaining can be overused — if a chain runs more than 5–6 steps or a step's logic is genuinely complex, breaking it into named intermediate steps with a comment is more readable, not less. Prioritize clarity over cleverness.

`pd.to_numeric()` is the standard tool for exactly the "sales column loaded as text" problem shown here — an extremely common occurrence when source data contains stray formatting (a currency symbol, a thousands comma, or even one single non-numeric row) that causes Pandas to fall back to loading the whole column as text (`dtype: object`) rather than a number.

### Cheat Sheet — Pandas Advanced

| Task | Code |
|---|---|
| Date range | `pd.date_range(start, periods=n, freq="D"/"W"/"M"/"Q")` |
| Rolling window | `.rolling(n).mean()` / `.sum()` |
| Expanding window | `.expanding().mean()` |
| % change | `.pct_change()` |
| Long → wide | `.pivot_table(index, columns, values)` |
| Wide → long | `.melt(id_vars, var_name, value_name)` |
| Chained transforms | `df.assign(col=lambda d: ...).assign(col2=lambda d: ...)` |
| Fix numeric-as-text | `pd.to_numeric(series)` |
| Memory-efficient labels | `df["col"] = df["col"].astype("category")` |

### 🎯 Practice — Pandas Advanced

Building directly on the mini-project idea from the source material: take a daily transaction table and build a "sales-operations feature table" with month, week, rolling 7-day sales, rolling 30-day sales, region rank, and month-over-month growth.

In [ ]:
np.random.seed(1)
daily = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=90, freq="D"),
    "region": np.random.choice(["North", "South", "East", "West"], 90),
    "sales": np.random.randint(8000, 25000, 90)
})

features = (
    daily
    .assign(
        month=lambda d: d["date"].dt.month,
        week=lambda d: d["date"].dt.isocalendar().week,
        rolling_7d=lambda d: d.groupby("region")["sales"].transform(lambda s: s.rolling(7).mean()),
        rolling_30d=lambda d: d.groupby("region")["sales"].transform(lambda s: s.rolling(30).mean())
    )
)
region_totals = features.groupby("region")["sales"].sum().rank(ascending=False)
print(features.tail(8))
print("\nRegion rank by total sales:\n", region_totals)

💡 `.groupby("region")["sales"].transform(...)` is a pattern worth flagging on its own: unlike `.groupby(...).agg(...)`, which collapses each group down to one summary row, `.transform()` returns a result the *same length as the original DataFrame* — here, a rolling average computed *separately within each region* but aligned back to every original row. This is exactly what you need when a per-group calculation should become a new column on the un-collapsed data, rather than a separate summary table.

---